# Demand Estimation and Market Analysis: Air Fryers

In this lab, you will study the market for air fryers using brand-year data aggregated from Amazon purchases. The goal is to move from descriptive analysis to a simple demand model, and then use that model to infer markups and unit costs.

Use the cleaned file:

```python
air_fryers_clean_brand_year.csv
```

This file keeps the top 10 air-fryer brands from 2019-2023 and drops the long tail of very small brands. The variable `brand_share` has already been recomputed within this cleaned name-brand market, so shares sum to 1 within each year.

## 2. Demand Estimation

We will estimate a logit-style demand model using linear regression. The model is:

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}p_{bt} + \beta_{rating}r_{bt} + \sum_{\ell=1}^L \beta_\ell x_{bt\ell} + \epsilon_{bt}.
$$

Here:

- $b$ indexes brands
- $t$ indexes years
- $s_{bt}$ is `brand_share`
- $p_{bt}$ is `avg_price`
- $r_{bt}$ is `avg_rating`
- $x_{bt\ell}$ are the product characteristics
- $\alpha_t$ are year dummy coefficients
- $\gamma_b$ are brand dummy coefficients
- $\beta_{price}$ is **one constant price coefficient**, shared by all brands and all years

That last point matters: do **not** estimate a different price coefficient for every brand-year. We do not have enough information for that, and it would make the cost calculation impossible to interpret.

Use `pd.get_dummies(..., drop_first=True)` for brand and year dummies. The dropped brand and dropped year become the reference categories, so all dummy coefficients are interpreted relative to those omitted categories.

Questions:

1. What is the estimated price coefficient, $\hat{\beta}_{price}$?
2. Is it negative? Why is that important?
3. Which product features are associated with higher demand?
4. Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand.
5. Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year.
6. What is the model's $R^2$?

This part of the work is the **data scientist** role: turning the cleaned data into a model that can be used for prediction and interpretation.

In [ ]:
#Q-2-1

# import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Create the dependent variable (log of brand share)
# Note: ensure there are no zero shares before taking the log,
# or you will get -inf values.
y = np.log(df['brand_share'])

# 2. Create dummy variables for brand and year
# drop_first=True prevents perfect multicollinearity (the dummy variable trap)
df_dummies = pd.get_dummies(df, columns=['brand', 'year'], drop_first=True)

# 3. Define the independent variables (X)
# Replace the list below with your actual product characteristic column names
product_features = [
    'compact_share',
    'dual_basket_share',
    'oven_style_share',
    'rotisserie_share',
    'window_share'
]

# Select our core variables + the generated dummies + product features
# Note: df_dummies contains the original columns PLUS the new dummy columns,
# but minus the original 'brand' and 'year' columns.
features_to_include = ['avg_price', 'avg_rating'] + product_features
dummy_cols = [col for col in df_dummies.columns if col.startswith('brand_') or col.startswith('year_')]

X = df_dummies[features_to_include + dummy_cols]

# Ensure all data in X is numeric (boolean dummies should be cast to float/int)
X = X.astype(float)

# 4. Add a constant (for alpha_0 in your equation)
X = sm.add_constant(X)

# 5. Fit the OLS (Ordinary Least Squares) model
model = sm.OLS(y, X).fit()

# 6. Print the results to answer your questions
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:            brand_share   R-squared:                       0.891
Model:                            OLS   Adj. R-squared:                  0.809
Method:                 Least Squares   F-statistic:                     10.91
Date:                Sun, 26 Apr 2026   Prob (F-statistic):           1.52e-08
Time:                        19:10:47   Log-Likelihood:                -17.052
No. Observations:                  50   AIC:                             78.10
Df Residuals:                      28   BIC:                             120.2
Df Model:                          21                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -5.8414     11.31

# Answer to Q-2-1

The estimated price coefficient is −0.0267.

# Answer to Q-2-2

Yes, it is negative. This is important because it aligns with the Law of Demand. A negative coefficient indicates an inverse relationship: as the price of the product increases, the market share decreases (holding all other factors constant). Statistically, this is highly significant in your model with a p-value of 0.002, which is well below the standard 0.05 threshold.

# Answer to Q-2-3

To find the features associated with higher demand, we need to look at positive coefficients:

**Dual Basket Share:** Has the largest positive impact with a coefficient of 12.1955.

**Compact Share:** Coefficient of 1.5448.

**Rotisserie Share:** Coefficient of 1.3326.



While these have positive coefficients, their p-values (all >0.5) suggest they are not "statistically significant" in this specific model. However, based purely on the direction of the coefficient, dual_basket_share is the strongest predictor of higher demand among features.

# Answer to Q-2-4

The brand coefficients are interpreted relative to the "dropped" brand:

**Brand Cuisinart:** This is the largest brand effect with a coefficient of 3.7536. It is also the only brand dummy that is statistically significant (p = 0.042).

**Brand Oster:** The next largest with a coefficient of 2.2191.

# Answer to Q-2-5

The year coefficients represent the baseline demand relative to the dropped year (likely 2019).

**Year 2021:** This year has the largest coefficient at 0.5052.

**Year 2023:** The next largest at 0.4449.

**Year 2022:** Close behind at 0.4379.

Demand appears to have peaked in 2021 and remained relatively stable (and higher than the base year) through 2023.

# Answer to Q-2-6

The R^2 is 0.891. This means that approximately 89.1% of the variance in brand share is explained by the variables included in your model, indicating a very strong fit for the data.